<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 02a — Vertex AI Experiments & Artifacts

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~0h40

</div>

## 📋 Lab Overview

In Lab 01 you used **MLflow** to track experiments locally. In production, teams need a managed, cloud-native solution that integrates with the rest of the ML platform. **Vertex AI Experiments** is Google Cloud's answer: it provides experiment tracking, artifact lineage, and metadata management — all accessible through the Vertex AI Python SDK.

### Learning Objectives

1. Initialize the Vertex AI Python SDK and connect to a GCP project.
2. Create an experiment and log parameters / metrics across multiple runs.
3. Compare runs programmatically using `get_experiment_df()`.
4. Register dataset and model **artifacts** and build an **artifact lineage** graph.
5. Visualize lineage in the Vertex AI console.
6. Clean up cloud resources.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install SDK, imports, GCP configuration |
| 1 | Create an Experiment | Initialize an experiment and start a run |
| 2 | Log Parameters & Metrics | Record hyperparameters and evaluation metrics |
| 3 | Compare Runs | Add a second run and compare results |
| 4 | Artifact Lineage | Create dataset/model artifacts and link them |
| 5 | Visualize & Inspect | View lineage in the console and extract results |
| 6 | Cleanup | Delete experiments and artifacts |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 82.5 MB/s eta 0:00:00


### 0.2 Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import google.cloud.aiplatform as aiplatform

print(f"Vertex AI SDK version: {aiplatform.__version__}")

Vertex AI SDK version: 1.140.0


### 0.3 Configuration

Replace the placeholders below with your own GCP project details.

In [6]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"          # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab02"  # @param {type:"string"}

# Initialize the Vertex AI SDK
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

✅ Vertex AI initialized — project=isae-sdd-481407, location=europe-west3


> 📖 **Docs:** [`aiplatform.init()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_init)

---
## 1 · Create an Experiment

A Vertex AI **Experiment** groups multiple training runs so you can compare them. The workflow is:

1. `aiplatform.init(experiment=...)` — create or attach to an experiment.
2. `aiplatform.start_run(...)` — start tracking a specific run.
3. Log parameters, metrics, artifacts.
4. `aiplatform.end_run()` — close the run.

> 📖 **Docs:** [Vertex AI Experiments overview](https://cloud.google.com/vertex-ai/docs/experiments/intro-vertex-ai-experiments)

In [7]:
##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = 'mregaieg'  # TODO
####################################################################

EXPERIMENT_NAME = f"supaero-lab02a-{your_name}"
print(f"Experiment name: {EXPERIMENT_NAME}")

Experiment name: supaero-lab02a-mregaieg


In [8]:
# Create the experiment and start the first run
aiplatform.init(experiment=EXPERIMENT_NAME)
aiplatform.start_run("run-1")
print("✅ Experiment created and run-1 started.")

✅ Experiment created and run-1 started.


---
## 2 · Log Parameters & Metrics

During a training run you typically want to record two categories of information:

**Parameters** describe the *configuration* of the run: architecture choices (number of units, layers), hyperparameters (learning rate, batch size), and data-processing decisions (split ratio, sampling strategy).

**Metrics** describe the *outcome*: evaluation scores, training time, early-stopping epoch, etc.

> 📖 **Key functions:**
> - [`aiplatform.log_params()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_log_params)
> - [`aiplatform.log_metrics()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_log_metrics)

In [9]:
# Log architecture parameters (meta-parameters)
aiplatform.log_params({"units": 128})

# Log training hyperparameters
aiplatform.log_params({
    "epochs": 100,
    "batch_size": 32,
    "learning_rate": 0.01,
})

# Log evaluation metrics (simulated values for now)
aiplatform.log_metrics({
    "train_acc": 97.3,
    "test_acc": 93.7,
})

print("✅ Parameters and metrics logged for run-1.")

✅ Parameters and metrics logged for run-1.


In [10]:
# End the run so we can start another one
aiplatform.end_run()

# Retrieve experiment results as a DataFrame
experiment_df = aiplatform.get_experiment_df()
experiment_df = experiment_df[experiment_df.experiment_name == EXPERIMENT_NAME]
print(experiment_df.T)

                                           0
experiment_name      supaero-lab02a-mregaieg
run_name                               run-1
run_type                system.ExperimentRun
state                               COMPLETE
param.learning_rate                     0.01
param.epochs                           100.0
param.batch_size                        32.0
param.units                            128.0
metric.train_acc                        97.3
metric.test_acc                         93.7


---
## 3 · Compare Runs

A key benefit of experiment tracking is the ability to compare runs side by side. Let's create a second run with different settings and see how the results change.

In [14]:
##############################  TODO  ##############################
# Start a new run called "run-2" on the same experiment.
# Change at least one parameter and adjust the metrics to simulate
# a training run that reduces overfitting (i.e., the gap between
# train_acc and test_acc should be smaller).
#
# Hint: you might increase units, adjust learning_rate, etc.

aiplatform.start_run("run-2")

aiplatform.log_params({
    "units": 256,           # TODO
    "epochs": 80,          # TODO
    "batch_size": 32,      # TODO
    "learning_rate": 0.005,   # TODO
})

aiplatform.log_metrics({
    "train_acc": 95.8,  # TODO — closer to test_acc
    "test_acc": 94.9,   # TODO — higher than run-1
})
####################################################################

print("✅ Parameters and metrics logged for run-3.")

✅ Parameters and metrics logged for run-2.


In [15]:
# End run-2 and compare both runs
aiplatform.end_run()

experiment_df = aiplatform.get_experiment_df()
experiment_df = experiment_df[experiment_df.experiment_name == EXPERIMENT_NAME]
print(experiment_df.T)

                                           0                        1  \
experiment_name      supaero-lab02a-mregaieg  supaero-lab02a-mregaieg   
run_name                               run-3                    run-2   
run_type                system.ExperimentRun     system.ExperimentRun   
state                               COMPLETE                 COMPLETE   
param.learning_rate                    0.005                     0.05   
param.epochs                            80.0                    100.0   
param.units                            256.0                    128.0   
param.batch_size                        32.0                     32.0   
metric.train_acc                        95.8                     96.0   
metric.test_acc                         94.9                     94.0   

                                           2  
experiment_name      supaero-lab02a-mregaieg  
run_name                               run-1  
run_type                system.ExperimentRun  
state   

**✏️ Question 1 — Overfitting**

a) Looking at run-1, is there evidence of overfitting? How can you tell from the logged metrics?

b) What changes did you make in run-2 to mitigate it?

---
*Your answer:*

a) Yes, there is a gap between train and test accuracy, which is a sign of overfitting.

b) To reduce it, I increased `units` and lowered the `learning_rate`, which helped make the train/test gap smaller and improved `test_acc`.


---

---
## 4 · Artifact Lineage

Beyond parameters and metrics, Vertex AI can track the **lineage** of ML artifacts: which dataset was used as input, which model was produced as output, and which execution linked them together. This is critical for reproducibility and auditing in production.

> 📖 **Docs:** [Artifact Lineage](https://docs.cloud.google.com/vertex-ai/docs/experiments/track-executions-artifact)

### 4.1 Create a new experiment for lineage tracking

In [16]:
LINEAGE_EXPERIMENT = f"supaero-lab02a-lineage-{your_name}"
aiplatform.init(experiment=LINEAGE_EXPERIMENT)
aiplatform.start_run("lineage-run-1")
print(f"✅ Lineage experiment '{LINEAGE_EXPERIMENT}' created.")

✅ Lineage experiment 'supaero-lab02a-lineage-mregaieg' created.


### 4.2 Create dataset & model artifacts

We create two synthetic artifacts to simulate a real training pipeline. In a real scenario these URIs would point to actual files in Cloud Storage.

> 📖 **Docs:** [`aiplatform.Artifact.create()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Artifact#google_cloud_aiplatform_Artifact_create)

In [17]:
DATASET_URI = f"gs://data/example_dataset.csv"
MODEL_URI = f"gs://models/example_model.pkl"

dataset_artifact = aiplatform.Artifact.create(
    schema_title="system.Dataset",
    display_name="example-dataset",
    uri=DATASET_URI,
)

model_artifact = aiplatform.Artifact.create(
    schema_title="system.Model",
    display_name="example-model",
    uri=MODEL_URI,
)

print(f"✅ Created dataset artifact: {dataset_artifact.display_name}")
print(f"✅ Created model artifact:   {model_artifact.display_name}")

✅ Created dataset artifact: example-dataset
✅ Created model artifact:   example-model


### 4.3 Build the lineage graph

An **execution** connects input artifacts to output artifacts. The procedure is:

1. `aiplatform.start_execution()` — create an execution context.
2. `execution.assign_input_artifacts([...])` — declare inputs.
3. Log parameters and metrics for this execution.
4. `execution.assign_output_artifacts([...])` — declare outputs.

> 📖 **Docs:** [`aiplatform.start_execution()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_start_execution)

In [20]:
##############################  TODO  ##############################
# Inside the execution context:
# 1. Log at least 2 hyperparameters using aiplatform.log_params()
# 2. Log at least 1 metric using aiplatform.log_metrics()

with aiplatform.start_execution(
    schema_title="system.ContainerExecution",
    display_name="example-training",
) as execution:
    execution.assign_input_artifacts([dataset_artifact])

    # Log hyperparameters for this execution
    aiplatform.log_params({
        "epochs": 100,
    "batch_size": 32,
    "learning_rate": 0.01,})  # TODO

    # Log evaluation metrics
    aiplatform.log_metrics({
         "train_acc": 97.3,
        "test_acc": 93.7,
    })  # TODO

    execution.assign_output_artifacts([model_artifact])
####################################################################

print("✅ Lineage graph created: dataset → execution → model")

✅ Lineage graph created: dataset → execution → model


---
## 5 · Visualize & Inspect

### 5.1 Extract experiment results

In [21]:
aiplatform.end_run()

lineage_df = aiplatform.get_experiment_df()
lineage_df = lineage_df[lineage_df.experiment_name == LINEAGE_EXPERIMENT]
print(lineage_df.T)

                                                   0
experiment_name      supaero-lab02a-lineage-mregaieg
run_name                               lineage-run-1
run_type                        system.ExperimentRun
state                                       COMPLETE
param.learning_rate                             0.01
param.epochs                                   100.0
param.batch_size                                32.0
metric.train_acc                                97.3
metric.test_acc                                 93.7


### 5.2 Open the lineage graph in the console

> 💡 **Tip:** Click the link below to see the artifact lineage graph in the Vertex AI console. You should see the dataset artifact connected to the model artifact through the execution.

In [22]:
lineage_url = execution.get_output_artifacts()[0].lineage_console_uri
print(f"Open the lineage graph: {lineage_url}")

Open the lineage graph: https://console.cloud.google.com/vertex-ai/locations/europe-west3/metadata-stores/default/artifacts/bea5e8ae-e726-4d48-93cf-f77d210d565f?project=isae-sdd-481407


**✏️ Question 2 — Lineage value**

a) In a production ML system, why is artifact lineage important? Give two concrete benefits.

b) What information would you add to the lineage to make it more useful for debugging a model regression?

---
*Your answer:*

a) Artifact lineage is important because it provides traceability across the ML pipeline and makes debugging much faster. It lets you see exactly which dataset, model, and execution produced a given artifact, and it helps identify what changed when performance drops.

b) To make the lineage more useful for debugging a regression, we would add the hyperparameters used for the run as well as the evaluation metrics, so it is easier to compare runs and spot what caused the regression.

---

---
## 6 · Cleanup

Always clean up cloud resources when you are done to avoid unnecessary charges.

In [23]:
# Delete artifacts
for artifact in [dataset_artifact, model_artifact]:
    try:
        artifact.delete()
        print(f"✅ Deleted artifact: {artifact.display_name}")
    except Exception as e:
        print(f"Could not delete {artifact.display_name}: {e}")

✅ Deleted artifact: example-dataset
✅ Deleted artifact: example-model


In [24]:
# Delete experiments
for exp_name in [EXPERIMENT_NAME, LINEAGE_EXPERIMENT]:
    try:
        exp = aiplatform.Experiment(exp_name)
        exp.delete(delete_backing_tensorboard_runs=True)
        print(f"✅ Deleted experiment: {exp_name}")
    except Exception as e:
        print(f"Could not delete {exp_name}: {e}")

✅ Deleted experiment: supaero-lab02a-mregaieg
✅ Deleted experiment: supaero-lab02a-lineage-mregaieg


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Setup | Initialized the Vertex AI SDK | `aiplatform.init()` |
| Experiment | Created an experiment and started runs | `aiplatform.init(experiment=...)`, `start_run()` |
| Logging | Recorded parameters and metrics | `log_params()`, `log_metrics()` |
| Comparison | Compared two runs side by side | `get_experiment_df()` |
| Lineage | Built a dataset → execution → model lineage graph | `Artifact.create()`, `start_execution()` |
| Visualization | Inspected lineage in the Vertex AI console | Console UI |
| Cleanup | Deleted all cloud resources | `delete()` methods |

**Next lab:** You will learn how to use the **Vertex AI Model Registry** to version, manage, and deploy models behind endpoints.